# Scheduling Optimization Solved with OpenJij (SQA Sampler)

In this notebook, we solve the following formulation
(with $\lambda_4, \lambda_5$ ignored)
by directly constructing the QUBO matrix.

## Formulation used

$$
\begin{aligned}
E
=&
-\alpha \sum_{i=1}^{I}\sum_{k=1}^{K}\sum_{t=1}^{T} C_{i,k}\,x_{i,k,t}
-\beta \sum_{i=1}^{I}\sum_{k=1}^{K}\sum_{t=1}^{T} T_{i,k}\,x_{i,k,t} \\
&+ \lambda_1 \sum_{i=1}^{I}\sum_{t=1}^{T}\left(\sum_{k=1}^{K}x_{i,k,t}-1\right)^2 \\
&+ \lambda_2 \sum_{k=1}^{K}\left(\sum_{i=1}^{I}\sum_{t=1}^{T}x_{i,k,t}-W_k\right)^2 \\
&+ \lambda_3 \sum_{i=1}^{I}\sum_{k=1}^{K}\sum_{t=1}^{T-1}\left(x_{i,k,t}-x_{i,k,t+1}\right)^2 \\
&+ \lambda_6 \sum_{i=1}^{I}\sum_{t=1}^{T}\sum_{k=1}^{K}(1-D_{i,t})x_{i,k,t}
\end{aligned}
$$

### Meaning of the variables

- $x_{i,k,t}\in\{0,1\}$: 1 if worker $i$ is assigned to task $k$ at time $t$
- $C_{i,k}$: interest between worker $i$ and task $k$.
- $T_{i,k}$: experience score of worker $i$ for task $k$.
- $W_k$: total required working time for task $k$
- $D_{i,t}\in\{0,1\}$: 1 if worker $i$ is available at time $t$

### Meaning of each term

1. Prefer combinations with high $C_{i,k}$  
2. Prefer combinations with high $T_{i,k}$  
3. Each worker is assigned to **exactly one task** at each time  
4. Satisfy the required total time $W_k$ for each task  
5. If the same worker continues the same task, prefer consecutive assignment when possible  
6. Avoid assignments during unavailable time slots  

> Note: The $\lambda_4,\lambda_5$ terms are not included in this notebook.


In [ ]:
# Run this only once at the beginning if needed
!pip install openjij matplotlib numpy


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import openjij as oj


## 1. Define the problem size and sample data

For beginners, we start with a small example that can be run as-is.  
Replace `I, K, T` and `C, T_score, W, D` with values for your own problem.


In [ ]:
# -----------------------------
# Problem size
# -----------------------------
I = 3  # number of workers
K = 3  # number of tasks
T = 6  # number of time slots

workers = [f"Worker{i}" for i in range(I)]
tasks   = [f"Task{k}" for k in range(K)]
times   = list(range(T))

# -----------------------------
# Example data
# C[i, k] : degree of interest
# T_score[i, k] : degree of experience
# W[k] : total required time for task k
# D[i, t] : 1 if available to work, 0 otherwise
# -----------------------------
C = np.array([
    [5, 1, 3],
    [3, 2, 1],
    [2, 5, 4]
], dtype=float)

T_score = np.array([
    [5, 2, 4],
    [2, 5, 5],
    [4, 2, 5]
], dtype=float)

W = np.array([3, 6, 4], dtype=int)   # total 18 slots = I*T = 3*6

D = np.array([
    [1, 1, 1, 1, 0, 1],  # Worker0
    [0, 1, 1, 1, 1, 1],  # Worker1
    [1, 1, 0, 1, 1, 1]   # Worker2
], dtype=int)

print("C =")
print(C)
print("\nT_score =")
print(T_score)
print("\nW =", W)
print("\nD =")
print(D)
print("\nTotal required time =", W.sum(), " / Total worker-time slots =", I*T)


## 2. Set the weights (hyperparameters)

As a basic rule, the $\lambda$ values for constraints should be set sufficiently larger than the objective coefficients $\alpha,\beta$.

Let’s try adjusting the hyperparameters.
If they are tuned well, we can obtain a solution that satisfies the constraints.


In [ ]:
alpha = 1.0
beta  = 1.0

#Enter the parameters for the objective function here

lam1 =   # each worker takes exactly one task at each time
lam2 =    # satisfy the required total time of each task
lam3 =     # prefer consecutive assignment when possible
lam6 =   # strongly prohibit assignment during unavailable time


In [ ]:
'''#One of the correct parameter settings
alpha = 1.0
beta  = 1.0

lam1 =800.0   
lam2 = 1000.0   
lam3 = 3.0   
lam6 = 3000.0   
'''

## 3. Assign indices to the variables $x_{i,k,t}$

Since the QUBO matrix is handled with one-dimensional variable indices,  
we create a mapping between $(i,k,t)$ and a one-dimensional index.


In [ ]:
# Mapping from (i, k, t) to q
index_map = {}
reverse_map = {}

q = 0
for i in range(I):
    for k in range(K):
        for t in range(T):
            index_map[(i, k, t)] = q
            reverse_map[q] = (i, k, t)
            q += 1

N = q  # total number of variables
print("Number of variables N =", N)
print("Example: index of x_(0,0,0) =", index_map[(0,0,0)])


## 4. Initialize the QUBO matrix with zeros

A QUBO is written as

$$
E(\mathbf{x}) = \sum_{i,j} Q_{i,j} x_i x_j
$$

Here, for convenience, we first create an $N \times N$ matrix Q.


In [ ]:
Q = np.zeros((N, N), dtype=float)

def add_qubo(q1, q2, value):
    # Helper function to add a value to the QUBO matrix Q
    # Accumulate values into the upper-triangular part where q1 <= q2
    if q1 <= q2:
        Q[q1, q2] += value
    else:
        Q[q2, q1] += value


## 5. Add the objective terms
$$
-\alpha \sum C_{i,k}x_{i,k,t}
-\beta \sum T_{i,k}x_{i,k,t}
$$

These are linear terms, so they are added to the diagonal entries of the QUBO matrix.


In [ ]:
# Objective function (linear terms)
for i in range(I):
    for k in range(K):
        for t in range(T):
            q = index_map[(i, k, t)]
            add_qubo(q, q, -alpha * C[i, k])
            add_qubo(q, q, -beta  * T_score[i, k])


## 6. Constraint 1: each worker is assigned to exactly one task at each time

$$
\lambda_1 \sum_{i,t} \left( \sum_k x_{i,k,t} - 1 \right)^2
$$

After expansion:

- the diagonal terms get $-\lambda_1$
- each pair of different tasks gets $2\lambda_1$



In [ ]:
for i in range(I):
    for t in range(T):
        # [x_{i,0,t}, x_{i,1,t}, ..., x_{i,K-1,t}]
        vars_it = [index_map[(i, k, t)] for k in range(K)]

        # (sum x - 1)^2 = sum x + 2sum_{a<b} x_a x_b - 2sum x + 1
        #                = -sum x + 2sum_{a<b} x_a x_b + 1
        # Constant terms are unnecessary in QUBO
        for q1 in vars_it:
            add_qubo(q1, q1, -lam1)

        for a in range(len(vars_it)):
            for b in range(a + 1, len(vars_it)):
                add_qubo(vars_it[a], vars_it[b], 2 * lam1)


## 7. Constraint 2: satisfy the required total time of each task

$$
\lambda_2 \sum_k \left( \sum_{i,t} x_{i,k,t} - W_k \right)^2
$$

When $(\sum z - W)^2$ is expanded:

- the diagonal terms get $(1 - 2W)\lambda_2$
- each pair of different variables gets $2\lambda_2$



In [ ]:
for k in range(K):
    vars_k = [index_map[(i, k, t)] for i in range(I) for t in range(T)]

    for q1 in vars_k:
        add_qubo(q1, q1, lam2 * (1 - 2 * W[k]))

    for a in range(len(vars_k)):
        for b in range(a + 1, len(vars_k)):
            add_qubo(vars_k[a], vars_k[b], 2 * lam2)


## 8. Constraint 3: prefer consecutive assignment to the same task
$$
\lambda_3 \sum_{i,k,t=1}^{T-1}(x_{i,k,t} - x_{i,k,t+1})^2
$$

$$
(x_t - x_{t+1})^2 = x_t + x_{t+1} - 2x_t x_{t+1}
$$

So we add:

- $+\lambda_3$ to the diagonal
- $-2\lambda_3$ to each adjacent-time pair



In [ ]:
for i in range(I):
    for k in range(K):
        for t in range(T - 1):
            q_now  = index_map[(i, k, t)]
            q_next = index_map[(i, k, t + 1)]

            add_qubo(q_now,  q_now,  lam3)
            add_qubo(q_next, q_next, lam3)
            add_qubo(q_now, q_next, -2 * lam3)


## 9. Constraint 4: avoid assignment during unavailable time slots
$$
\lambda_6 \sum_{i,t,k}(1-D_{i,t})x_{i,k,t}
$$

This is also a linear term, so it can be added to the diagonal entries.


In [ ]:
for i in range(I):
    for t in range(T):
        for k in range(K):
            q = index_map[(i, k, t)]
            add_qubo(q, q, lam6 * (1 - D[i, t]))


## 10. Convert to the QUBO dictionary format for OpenJij

In OpenJij, the QUBO can be passed as a dictionary of the form `{(p, q): value}`.  
We store only the non-zero elements.


In [ ]:
qubo = {}
for i in range(N):
    for j in range(i, N):
        if abs(Q[i, j]) > 1e-12:
            qubo[(i, j)] = Q[i, j]

print("Number of QUBO terms =", len(qubo))


## 11. Solve with the SQA sampler


In [ ]:
sampler = oj.SQASampler()
response = sampler.sample_qubo(qubo, num_reads=200)

best_sample = response.first.sample
best_energy = response.first.energy

print("best energy =", best_energy)


## 12. Convert the solution back to the 3D array $x_{i,k,t}$


In [ ]:
x_sol = np.zeros((I, K, T), dtype=int)

for q, value in best_sample.items():
    i, k, t = reverse_map[q]
    x_sol[i, k, t] = value

print("x_sol.shape =", x_sol.shape)


## 13. Print the solution in an easy-to-read format

For each worker and time, display which task has been assigned.


In [ ]:
assignment_table = []

for i in range(I):
    row = []
    for t in range(T):
        assigned_tasks = [k for k in range(K) if x_sol[i, k, t] == 1]

        if len(assigned_tasks) == 1:
            row.append(f"Task{assigned_tasks[0]}")
        elif len(assigned_tasks) == 0:
            row.append("-")
        else:
            row.append(" / ".join([f"Task{k}" for k in assigned_tasks]))

    assignment_table.append(row)

print("=== Assignment result (rows: Worker, columns: Time) ===")
for i in range(I):
    print(f"{workers[i]}: {assignment_table[i]}")


## 14. Also check the total assigned time for each task


In [ ]:
print("=== Total assigned time for each task ===")
for k in range(K):
    total_time = x_sol[:, k, :].sum()
    print(f"Task{k}: {total_time}  (required: {W[k]})")


## 15. Draw a Gantt chart

- Vertical axis: Worker  
- Horizontal axis: Time  
- Color: Task  

If multiple tasks are assigned at the same time, multiple short bars are drawn within that worker's time slot.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

cmap = plt.cm.get_cmap("tab10", K)

for i in range(I):
    y = i
    for t in range(T):
        assigned_tasks = [k for k in range(K) if x_sol[i, k, t] == 1]

        if len(assigned_tasks) == 0:
            continue

        h = 0.8 / len(assigned_tasks)

        for idx, k in enumerate(assigned_tasks):
            ax.broken_barh(
                [(t, 1)],
                (y - 0.4 + idx * h, h),
                facecolors=cmap(k),
                edgecolors="black"
            )
            ax.text(
                t + 0.5,
                y - 0.4 + idx * h + h / 2,
                f"Task{k}",
                ha="center",
                va="center",
                fontsize=8
            )

ax.set_xlim(0, T)
ax.set_ylim(-0.8, I - 0.2)
ax.set_xticks(range(T + 1))
ax.set_yticks(range(I))
ax.set_yticklabels(workers)
ax.set_xlabel("Time")
ax.set_ylabel("Worker")
ax.set_title("Scheduling result (Gantt chart)")
ax.grid(True)

handles = []
labels = []
for k in range(K):
    patch = plt.Rectangle((0, 0), 1, 1, fc=cmap(k), ec="black")
    handles.append(patch)
    labels.append(f"Task{k}")

ax.legend(handles, labels, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


# Additional Example: Number Partitioning Problem

In this section, we add another simple optimization example using OpenJij: the **number partitioning problem**.

The goal is to divide a set of numbers into two groups so that the difference between the sums of the two groups becomes as small as possible.

This problem is useful for beginners because:
- the idea is intuitive,
- the QUBO formulation is simple,
- and we can build the QUBO matrix directly with loops, just like in the scheduling example.


## 16. Formulation of the number partitioning problem

Suppose the given numbers are:

$$
a_1, a_2, \dots, a_N
$$

Introduce a binary variable:
$$
x_i =
\begin{cases}
1 & \text{if number } a_i \text{ is assigned to Group A} \\
0 & \text{if number } a_i \text{ is assigned to Group B}
\end{cases}
$$

Then the sum of Group A is:

$$
\sum_{i=1}^{N} a_i x_i
$$

and the sum of Group B is:

$$
\sum_{i=1}^{N} a_i (1 - x_i)
$$

Therefore, the difference between the two groups is:

$$
\sum_{i=1}^{N} a_i x_i - \sum_{i=1}^{N} a_i (1 - x_i)
= \sum_{i=1}^{N} a_i (2x_i - 1)
$$

We minimize the square of this difference:

$$
E = \left( \sum_{i=1}^{N} a_i (2x_i - 1) \right)^2
$$

By minimizing this energy, we obtain a partition where the two sums are as close as possible.


In [ ]:
# Example data for the number partitioning problem
# We want to divide these numbers into two groups with nearly equal sums.

numbers = np.array([8, 7, 6, 5, 4, 3, 1])
N = len(numbers)

print("Numbers:", numbers)
print("Total sum:", numbers.sum())


## 17. Build the QUBO matrix directly with loops

Here we construct the QUBO matrix for

$$
E = \left( \sum_{i=1}^{N} a_i (2x_i - 1) \right)^2 .
$$

To keep the implementation beginner-friendly, we first rewrite the expression as

$$
E = \left( 2 \sum_i a_i x_i - S \right)^2,
\qquad \text{where} \qquad
S = \sum_i a_i
$$

and then expand it term by term into a QUBO matrix.

In [ ]:
# Build the QUBO matrix Q
# The energy is handled as: E = sum_{i,j} Q[i, j] x_i x_j + const

S = numbers.sum()
Q_np = np.zeros((N, N))
const_np = S**2

# Diagonal terms
for i in range(N):
    # From 4 * (a_i^2) * x_i^2 - 4 * S * a_i * x_i
    # Since x_i^2 = x_i for binary variables, both contributions go to the diagonal.
    Q_np[i, i] += 4 * numbers[i] * numbers[i] - 4 * S * numbers[i]

# Off-diagonal terms
for i in range(N):
    for j in range(i + 1, N):
        # From 8 * a_i * a_j * x_i * x_j
        Q_np[i, j] += 8 * numbers[i] * numbers[j]

# Convert the upper-triangular matrix into a dictionary format for OpenJij
qubo_np = {}
for i in range(N):
    for j in range(i, N):
        if Q_np[i, j] != 0:
            qubo_np[(i, j)] = Q_np[i, j]

print("QUBO dictionary:")
for key, value in qubo_np.items():
    print(f"{key}: {value}")
print("Constant term:", const_np)


## 18. Solve the QUBO with the OpenJij SQA sampler

As in the scheduling example, we use the simulated quantum annealing sampler in OpenJij to search for a low-energy solution.


In [ ]:
sampler_np = oj.SQASampler()
response_np = sampler_np.sample_qubo(qubo_np, num_reads=100)
best_sample_np = response_np.first.sample
best_energy_np = response_np.first.energy

print("Best sample:", best_sample_np)
print("Best energy:", best_energy_np)


## 19. Decode the solution and check the result

Finally, we convert the binary solution into two groups and compare their sums.


In [ ]:
x_np = np.array([best_sample_np[i] for i in range(N)])

group_A = [numbers[i] for i in range(N) if x_np[i] == 1]
group_B = [numbers[i] for i in range(N) if x_np[i] == 0]

sum_A = sum(group_A)
sum_B = sum(group_B)
difference = abs(sum_A - sum_B)

print("=== Number Partitioning Result ===")
print("Group A:", group_A)
print("Group B:", group_B)
print("Sum of Group A:", sum_A)
print("Sum of Group B:", sum_B)
print("Difference:", difference)
